In [ ]:
pip install

In [ ]:
from kubernetes import client as k8s_client
from kubeflow.training import V1ReplicaSpec, V1RunPolicy
from kubeflow.training import client as training_client
from kubernetes.client.models import V1ObjectMeta, V1PodSpec, V1PodTemplateSpec
from kubeflow.training import V1TFJob, V1TFJobSpec, V1PyTorchJob, V1PyTorchJobSpec, V1MPIJob, V1MPIJobSpec

class TrainingJobSDK:
    def __init__(self, name, namespace, container_name, training_image, model_dir, data_dir, local_output_dir, script_dir):
        self.name = name
        self.namespace = namespace
        self.container_name = container_name
        self.training_image = training_image
        self.model_dir = model_dir
        self.data_dir = data_dir
        self.local_output_dir = local_output_dir
        self.script_dir = script_dir
        self.client = training_client.TrainingClient(namespace=self.namespace)

    def create_volume(self, name, host_path):
        return k8s_client.V1Volume(
            name=name,
            host_path=k8s_client.V1HostPathVolumeSource(path=host_path)
        )

    def create_volume_mount(self, name, mount_path):
        return k8s_client.V1VolumeMount(name=name, mount_path=mount_path)

    def create_pod_spec(self, script_name, args):
        container = k8s_client.V1Container(
            name=self.container_name,
            image=self.training_image,
            command=["python", self.script_dir + "/" + script_name] + args,
            volume_mounts=[
                self.create_volume_mount("model-volume", self.model_dir),
                self.create_volume_mount("data-volume", self.data_dir),
                self.create_volume_mount("local-output-volume", self.local_output_dir),
                self.create_volume_mount("script-volume", self.script_dir)
            ]
        )

        pod_spec = V1PodSpec(
            containers=[container],
            volumes=[
                self.create_volume("model-volume", self.model_dir),
                self.create_volume("data-volume", self.data_dir),
                self.create_volume("local-output-volume", self.local_output_dir),
                self.create_volume("script-volume", self.script_dir)
            ]
        )

        return pod_spec

    def create_replica_spec(self, replicas, pod_spec):
        return V1ReplicaSpec(
            replicas=replicas,
            restart_policy="Never",
            template=V1PodTemplateSpec(spec=pod_spec)
        )

    def create_training_job(self, job_type, replicas, script_name, args):
        pod_spec = self.create_pod_spec(script_name, args)

        if job_type == "TFJob":
            tfjob = V1TFJob(
                api_version="kubeflow.org/v1",
                kind="TFJob",
                metadata=V1ObjectMeta(name=self.name, namespace=self.namespace),
                spec=V1TFJobSpec(
                    run_policy=V1RunPolicy(clean_pod_policy="None"),
                    tf_replica_specs={
                        "Chief": self.create_replica_spec(replicas.get("Chief", 1), pod_spec),
                        "Worker": self.create_replica_spec(replicas.get("Worker", 2), pod_spec),
                        "PS": self.create_replica_spec(replicas.get("PS", 1), pod_spec)
                    }
                )
            )
            return tfjob

        elif job_type == "PyTorchJob":
            pytorchjob = V1PyTorchJob(
                api_version="kubeflow.org/v1",
                kind="PyTorchJob",
                metadata=V1ObjectMeta(name=self.name, namespace=self.namespace),
                spec=V1PyTorchJobSpec(
                    run_policy=V1RunPolicy(clean_pod_policy="None"),
                    pytorch_replica_specs={
                        "Master": self.create_replica_spec(replicas.get("Master", 1), pod_spec),
                        "Worker": self.create_replica_spec(replicas.get("Worker", 2), pod_spec)
                    }
                )
            )
            return pytorchjob

        elif job_type == "MPIJob":
            mpijob = V1MPIJob(
                api_version="kubeflow.org/v1",
                kind="MPIJob",
                metadata=V1ObjectMeta(name=self.name, namespace=self.namespace),
                spec=V1MPIJobSpec(
                    run_policy=V1RunPolicy(clean_pod_policy="None"),
                    mpi_replica_specs={
                        "Launcher": self.create_replica_spec(replicas.get("Launcher", 1), pod_spec),
                        "Worker": self.create_replica_spec(replicas.get("Worker", 2), pod_spec)
                    }
                )
            )
            return mpijob

        else:
            raise ValueError(f"Unsupported job type: {job_type}")

    def submit_job(self, job):
        if isinstance(job, V1TFJob):
            self.client.create_job(job, job.metadata.name)
        elif isinstance(job, V1PyTorchJob):
            self.client.create_job(job, job.metadata.name)
        elif isinstance(job, V1MPIJob):
            self.client.create_job(job, job.metadata.name)
        else:
            raise ValueError("Unsupported job type for submission")

    def get_job_status(self):
        try:
            status = self.client.get_job(self.name)
            print(f"Job status: {status.status.conditions}")
        except Exception as e:
            print(f"Failed to get job status: {e}")

    def get_job_logs(self, follow=False, verbose=False):
        try:
            logs, events = self.client.get_job_logs(
                name=self.name,
                follow=follow,
                verbose=verbose
            )
            for pod_name, log in logs.items():
                print(f"Logs for {pod_name}: {log}")
            if verbose:
                for event_name, event_data in events.items():
                    print(f"Events for {event_name}: {event_data}")
        except Exception as e:
            print(f"Failed to get job logs: {e}")

    def update_job(self, updated_job):
        try:
            self.client.update_job(updated_job, name=self.name)
            print(f"Job '{self.name}' updated successfully.")
        except Exception as e:
            print(f"Failed to update job: {e}")

    def delete_job(self):
        try:
            self.client.delete_job(name=self.name)
            print(f"Job '{self.name}' deleted successfully.")
        except Exception as e:
            print(f"Failed to delete job: {e}")

    def get_job_pods(self):
        try:
            pods = self.client.get_job_pods(name=self.name)
            for pod in pods:
                print(f"Pod name: {pod.metadata.name}, Status: {pod.status.phase}")
        except Exception as e:
            print(f"Failed to get job pods: {e}")

# 使用示例
sdk = TrainingJobSDK(
    name="example-training-job",
    namespace="kubeflow-user-example-com",
    container_name="training-container",
    training_image="tensorflow/tensorflow:2.4.1",
    model_dir="/mnt/model",
    data_dir="/mnt/data",
    local_output_dir="/mnt/local_output",
    script_dir="/mnt/scripts"
)

# 定义 TFJob
tfjob = sdk.create_training_job(
    job_type="TFJob",
    replicas={"Chief": 1, "Worker": 2, "PS": 1},
    script_name="mnist_with_summaries.py",
    args=[
        "--log_dir=" + sdk.model_dir + "/logs",
        "--data_dir=" + sdk.data_dir,
        "--model_dir=" + sdk.model_dir,
        "--learning_rate=0.01",
        "--batch_size=150"
    ]
)

# 提交 TFJob
sdk.submit_job(tfjob)

# 获取 TFJob 状态
sdk.get_job_status()

# 获取 TFJob 日志
sdk.get_job_logs(follow=True, verbose=True)

# 更新 TFJob
# updated_tfjob = sdk.create_training_job(
#     job_type="TFJob",
#     replicas={"Chief": 1, "Worker": 3, "PS": 1},  # 示例：更新 Worker 数量
#     script_name="mnist_with_summaries.py",
#     args=[
#         "--log_dir=" + sdk.model_dir + "/logs",
#         "--data_dir=" + sdk.data_dir,
#         "--model_dir=" + sdk.model_dir,
#         "--learning_rate=0.01",
#         "--batch_size=150"
#     ]
# )
# sdk.update_job(updated_tfjob)

# 删除 TFJob
sdk.delete_job()

# 获取 TFJob 的 Pod 列表
sdk.get_job_pods()
